In [10]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [11]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

data = pd.read_csv(url, names=columns)

print(data.head())
print(data.shape)
print(data.info())
print(data.describe())
print(data["Outcome"].value_counts())

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
(768, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose             

In [12]:
X = data.drop("Outcome", axis=1)
y = data["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (614, 8)
Testing data: (154, 8)


In [13]:
class KerasModel(BaseEstimator, ClassifierMixin):

    def __init__(
        self,
        neurons=16,
        activation="relu",
        dropout_rate=0.0,
        learning_rate=0.001,
        kernel_initializer="glorot_uniform",
        batch_size=32,
        epochs=20
    ):
        self.neurons = neurons
        self.activation = activation
        self.dropout_rate = dropout_rate
        self.learning_rate = learning_rate
        self.kernel_initializer = kernel_initializer
        self.batch_size = batch_size
        self.epochs = epochs

    def build_model(self):
        model = Sequential([
            Dense(
                self.neurons,
                input_shape=(X_train.shape[1],),
                activation=self.activation,
                kernel_initializer=self.kernel_initializer
            ),
            Dropout(self.dropout_rate),
            Dense(
                self.neurons // 2,
                activation=self.activation,
                kernel_initializer=self.kernel_initializer
            ),
            Dropout(self.dropout_rate),
            Dense(1, activation="sigmoid")
        ])

        model.compile(
            optimizer=Adam(learning_rate=self.learning_rate),
            loss="binary_crossentropy",
            metrics=["accuracy"]
        )

        return model

    def fit(self, X, y):
        self.model_ = self.build_model()

        self.model_.fit(
            X,
            y,
            epochs=self.epochs,
            batch_size=self.batch_size,
            verbose=0
        )

        self.classes_ = np.array([0, 1])

        return self

    def predict(self, X):
        probabilities = self.model_.predict(X, verbose=0).ravel()
        return (probabilities >= 0.5).astype(int)

    def predict_proba(self, X):
        probabilities = self.model_.predict(X, verbose=0).ravel()

        return np.column_stack([
            1 - probabilities,
            probabilities
        ])

In [14]:
model = KerasModel()

param_grid = {
    "batch_size": [16, 32],
    "epochs": [20],
    "learning_rate": [0.001],
    "kernel_initializer": ["glorot_uniform"],
    "activation": ["relu", "tanh"],
    "dropout_rate": [0.0, 0.2],
    "neurons": [16, 32]
}

In [15]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='keras')

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=2,
    scoring="accuracy",
    n_jobs=1,
    verbose=2
)

grid_result = grid.fit(X_train, y_train)

Fitting 2 folds for each of 16 candidates, totalling 32 fits
[CV] END activation=relu, batch_size=16, dropout_rate=0.0, epochs=20, kernel_initializer=glorot_uniform, learning_rate=0.001, neurons=16; total time=   5.8s
[CV] END activation=relu, batch_size=16, dropout_rate=0.0, epochs=20, kernel_initializer=glorot_uniform, learning_rate=0.001, neurons=16; total time=   6.2s
[CV] END activation=relu, batch_size=16, dropout_rate=0.0, epochs=20, kernel_initializer=glorot_uniform, learning_rate=0.001, neurons=32; total time=   5.6s
[CV] END activation=relu, batch_size=16, dropout_rate=0.0, epochs=20, kernel_initializer=glorot_uniform, learning_rate=0.001, neurons=32; total time=   4.3s
[CV] END activation=relu, batch_size=16, dropout_rate=0.2, epochs=20, kernel_initializer=glorot_uniform, learning_rate=0.001, neurons=16; total time=   3.9s
[CV] END activation=relu, batch_size=16, dropout_rate=0.2, epochs=20, kernel_initializer=glorot_uniform, learning_rate=0.001, neurons=16; total time=   5.

In [16]:
print("Best Parameters:")
print(grid_result.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(grid_result.best_score_)

Best Parameters:
{'activation': 'tanh', 'batch_size': 16, 'dropout_rate': 0.2, 'epochs': 20, 'kernel_initializer': 'glorot_uniform', 'learning_rate': 0.001, 'neurons': 32}

Best Cross-Validation Accuracy:
0.7833876221498371


In [17]:
results = pd.DataFrame(grid_result.cv_results_)

results = results.sort_values(
    by="mean_test_score",
    ascending=False
)

results[
    [
        "param_batch_size",
        "param_epochs",
        "param_learning_rate",
        "param_kernel_initializer",
        "param_activation",
        "param_dropout_rate",
        "param_neurons",
        "mean_test_score",
        "std_test_score"
    ]
].head(10)

,param_batch_size,param_epochs,param_learning_rate,param_kernel_initializer,param_activation,param_dropout_rate,param_neurons,mean_test_score,std_test_score
11,16,20,0.001,glorot_uniform,tanh,0.2,32,0.783388,0.004886
13,32,20,0.001,glorot_uniform,tanh,0.0,32,0.781759,0.009772
15,32,20,0.001,glorot_uniform,tanh,0.2,32,0.781759,0.009772
9,16,20,0.001,glorot_uniform,tanh,0.0,32,0.778502,0.009772
12,32,20,0.001,glorot_uniform,tanh,0.0,16,0.775244,0.003257
3,16,20,0.001,glorot_uniform,relu,0.2,32,0.768730,0.003257
8,16,20,0.001,glorot_uniform,tanh,0.0,16,0.768730,0.006515
10,16,20,0.001,glorot_uniform,tanh,0.2,16,0.767101,0.001629
5,32,20,0.001,glorot_uniform,relu,0.0,32,0.763844,0.011401
14,32,20,0.001,glorot_uniform,tanh,0.2,16,0.760586,0.004886
